In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (confusion_matrix,classification_report,
                             precision_recall_curve,roc_curve, auc,
                             roc_auc_score, average_precision_score,
                             brier_score_loss, mean_squared_error,
                             mean_absolute_error, r2_score)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                      TimeSeriesSplit)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.datasets import make_classification
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)

In [2]:
y_true  = np.array([1]*10 + [0]*90)
y_pred  = np.array([1]*8  + [0]*2  + [1]*5 + [0]*85)

cm = confusion_matrix(y_true, y_pred)
TP, FN = cm[1,1], cm[1,0]
FP, TN = cm[0,1], cm[0,0]

print("=== Confusion Matrix Analysis ===")
print(f"  TP={TP} (caught cancer)      FP={FP} (false alarms)")
print(f"  FN={FN} (missed cancer!)     TN={TN} (correctly healthy)")
print(f"Accuracy:  {(TP+TN)/(TP+TN+FP+FN):.3f}  ← misleading! A model predicting ALL=0 gets 0.90!")
print(f"  Precision: {TP/(TP+FP):.3f}  → of all 'cancer' predictions, {TP/(TP+FP)*100:.0f}% were real")
print(f"  Recall:    {TP/(TP+FN):.3f}  → of all actual cancer cases, caught {TP/(TP+FN)*100:.0f}%")
print(f"  F1:        {2*TP/(2*TP+FP+FN):.3f}")
print(f"For cancer detection: RECALL is critical (missing cancer = death)")
print(f"  Missing {FN} cancer patients is catastrophic vs {FP} false alarms (more tests)")


=== Confusion Matrix Analysis ===
  TP=8 (caught cancer)      FP=5 (false alarms)
  FN=2 (missed cancer!)     TN=85 (correctly healthy)
Accuracy:  0.930  ← misleading! A model predicting ALL=0 gets 0.90!
  Precision: 0.615  → of all 'cancer' predictions, 62% were real
  Recall:    0.800  → of all actual cancer cases, caught 80%
  F1:        0.696
For cancer detection: RECALL is critical (missing cancer = death)
  Missing 2 cancer patients is catastrophic vs 5 false alarms (more tests)


In [3]:
X_imb, y_imb = make_classification(
    n_samples=10000, n_features=10, weights=[0.95, 0.05],
    n_informative=5, random_state=42
)
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.2,
                                           stratify=y_imb, random_state=42)

rf_imb = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                 n_jobs=-1, random_state=42)
rf_imb.fit(X_tr, y_tr)
y_prob_imb = rf_imb.predict_proba(X_te)[:,1]


fpr, tpr, roc_thresholds = roc_curve(y_te, y_prob_imb)
prec, rec, pr_thresholds  = precision_recall_curve(y_te, y_prob_imb)

roc_auc = auc(fpr, tpr)
pr_auc  = average_precision_score(y_te, y_prob_imb)
baseline_pr = y_te.mean()

print("=== AUC-ROC vs PR-AUC (5% positive class) ===")
print(f"  ROC-AUC:  {roc_auc:.4f}  ← looks great even for weak models")
print(f"  PR-AUC:   {pr_auc:.4f}  ← harder to score high when classes are skewed")
print(f"  Baseline PR-AUC (random): {baseline_pr:.4f}  ← the real floor")
print(f"→ For fraud/rare disease: PR-AUC is more informative than ROC-AUC")



=== AUC-ROC vs PR-AUC (5% positive class) ===
  ROC-AUC:  0.9552  ← looks great even for weak models
  PR-AUC:   0.8023  ← harder to score high when classes are skewed
  Baseline PR-AUC (random): 0.0540  ← the real floor
→ For fraud/rare disease: PR-AUC is more informative than ROC-AUC


In [4]:
print("=== Optimal Threshold Selection ===")
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>8} {'FP cost':>10}")

for thr in np.arange(0.2, 0.8, 0.1):
    y_pred_t = (y_prob_imb >= thr).astype(int)
    tp = ((y_pred_t==1) & (y_te==1)).sum()
    fp = ((y_pred_t==1) & (y_te==0)).sum()
    fn = ((y_pred_t==0) & (y_te==1)).sum()
    p = tp/(tp+fp) if (tp+fp)>0 else 0
    r = tp/(tp+fn) if (tp+fn)>0 else 0
    f1 = 2*p*r/(p+r) if (p+r)>0 else 0
    print(f"  {thr:>8.1f}   {p:>8.3f}   {r:>8.3f}   {f1:>6.3f}   {fp:>8}")


=== Optimal Threshold Selection ===
 Threshold  Precision     Recall       F1    FP cost
       0.2      0.672      0.815    0.736         43
       0.3      0.778      0.778    0.778         24
       0.4      0.838      0.620    0.713         13
       0.5      0.929      0.481    0.634          4
       0.6      0.930      0.370    0.530          3
       0.7      0.958      0.213    0.348          1
       0.8      1.000      0.102    0.185          0


In [5]:
from sklearn.datasets import make_classification
X_cv, y_cv = make_classification(n_samples=2000, n_features=10,
                                  class_sep=0.8, random_state=42)

rf_cv = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

# Standard k-fold (wrong for imbalanced!)
kfold_scores = cross_val_score(rf_cv, X_cv, y_cv, cv=5, scoring="roc_auc")

# Stratified k-fold (maintains class ratio per fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_scores = cross_val_score(rf_cv, X_cv, y_cv, cv=skf, scoring="roc_auc")

print(f"=== Cross-Validation Comparison ===")
print(f"  Standard KFold:   {kfold_scores.mean():.4f} ± {kfold_scores.std():.4f}")
print(f"  Stratified KFold: {skf_scores.mean():.4f} ± {skf_scores.std():.4f}")
print(f"  → Stratified is ALWAYS preferred for classification")



=== Cross-Validation Comparison ===
  Standard KFold:   0.9452 ± 0.0093
  Stratified KFold: 0.9456 ± 0.0128
  → Stratified is ALWAYS preferred for classification


In [6]:
y_true_reg = np.array([100, 200, 150, 80, 300, 250, 90, 175])
y_pred_reg = np.array([110, 190, 160, 85, 310, 230, 95, 170])

mae  = mean_absolute_error(y_true_reg, y_pred_reg)
mse  = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_true_reg - y_pred_reg)/y_true_reg)) * 100
r2   = r2_score(y_true_reg, y_pred_reg)

print(f"=== Regression Metrics ===")
print(f"  MAE:  {mae:.2f}  ← in same units as y, robust to outliers")
print(f"  RMSE: {rmse:.2f}  ← penalises large errors more (sensitive to outliers)")
print(f"  MAPE: {mape:.2f}% ← relative, but fails when y ≈ 0")
print(f"  R²:   {r2:.4f}  ← % variance explained (1.0 = perfect, <0 = worse than mean)")
print(f"Choose RMSE when: large errors are especially costly (pricing)")
print(f"  Choose MAE when:  outliers in y and you want robustness")
print(f"  Choose MAPE when: communicating to business (percentage-based)")

=== Regression Metrics ===
  MAE:  9.38  ← in same units as y, robust to outliers
  RMSE: 10.46  ← penalises large errors more (sensitive to outliers)
  MAPE: 5.96% ← relative, but fails when y ≈ 0
  R²:   0.9801  ← % variance explained (1.0 = perfect, <0 = worse than mean)
Choose RMSE when: large errors are especially costly (pricing)
  Choose MAE when:  outliers in y and you want robustness
  Choose MAPE when: communicating to business (percentage-based)
